# E14：变量关系、变化结构与 EDA 表达

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用散点图和颜色分组探索两个数值变量的关系。
2. 使用描述性趋势线并说明相关、推断和因果之间的边界。
3. 先对有序记录进行分组汇总，再用折线图表达变化。
4. 计算相关矩阵并使用热力图展示数值变量的相关结构。
5. 围绕一个问题组织三张图，形成简洁、可复现的 EDA 短报告。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、图形和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的官方分析视图。
4. 提交前重启内核并按顺序运行全部单元，检查数据口径、标题、坐标轴和文字结论。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的路径和输入准备单元，再完成后续任务。
- 修改作图代码后必须重新运行当前单元，图形才会更新。
- 图形异常时，先检查列名、数据类型、缺失值、类别顺序和当前使用的 `Axes`。
- 如果变量或图形状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 输入选择、核验和问题确认：10 分钟
- 散点图与描述性趋势：25 分钟
- 轮次汇总与折线图：20 分钟
- 相关矩阵与热力图：15 分钟
- 三图 EDA 表达：15 分钟
- 最终核验与提交：5 分钟

## 两种观测单位：学生与学生—轮次

`student_df` 一行代表一名学生，适合研究学生之间的分布和变量关系；`wave_df` 一行代表一名学生在一个轮次的记录，适合研究有序变化。准备代码优先复用 E12 和 E11 个人成果，并统一为本章字段；个人成果不存在时使用官方分析视图。不要把两张表的行数或 `python_score` 口径混为一谈。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# 有可靠个人成果时保持 True；需要从官方分析视图开始时改为 False。
USE_STUDENT_E12_OUTPUT = True
USE_STUDENT_E11_OUTPUT = True

chapter5_dir = Path(".").resolve()
notebook_names = ["e14.ipynb", "e14_inclass.ipynb"]
if not any((chapter5_dir / name).exists() for name in notebook_names):
    chapter5_dir = Path("lab/applied-stat-64/chapter5").resolve()

shared_dir = chapter5_dir.parents[1] / "shared"
chapter4_dir = chapter5_dir.parent / "chapter4"
student_e12_path = (
    chapter4_dir / "e12_output" / "e12_analysis_ready.csv"
)
student_e11_path = (
    chapter4_dir
    / "e11_output"
    / "e11_learning_records_clean.csv"
)
official_student_path = shared_dir / "analysis" / "student_eda.csv"
official_wave_path = shared_dir / "analysis" / "learning_wave.csv"
output_dir = chapter5_dir / "e14_output"

student_columns = [
    "student_id",
    "gender",
    "major",
    "study_hours",
    "attendance_rate",
    "python_score",
    "data_literacy_score",
]
major_order = [
    "applied_statistics",
    "statistical_computing",
    "data_science",
]

if USE_STUDENT_E12_OUTPUT and student_e12_path.exists():
    raw_students = pd.read_csv(student_e12_path)
    required = [
        "student_id",
        "gender",
        "major",
        "study_hours",
        "attendance_rate",
        "final_score",
        "data_literacy_score",
    ]
    missing = [column for column in required if column not in raw_students]
    if missing:
        raise ValueError(
            f"E12 个人成果缺少字段：{missing}。"
            "请修正个人成果，或关闭对应输入开关。"
        )
    student_df = (
        raw_students[required]
        .rename(columns={"final_score": "python_score"})
        .copy()
    )
    print("学生表来源：E12 个人成果")
else:
    if USE_STUDENT_E12_OUTPUT:
        print("未找到 E12 个人成果，改用官方学生分析视图。")
    student_df = pd.read_csv(official_student_path)
    print("学生表来源：官方分析视图")

if USE_STUDENT_E11_OUTPUT and student_e11_path.exists():
    raw_wave = pd.read_csv(student_e11_path)
    required = ["student_id", "wave", "study_hours", "quiz_score"]
    missing = [column for column in required if column not in raw_wave]
    if missing:
        raise ValueError(
            f"E11 个人成果缺少字段：{missing}。"
            "请修正个人成果，或关闭对应输入开关。"
        )
    wave_df = (
        raw_wave[required]
        .merge(
            student_df[["student_id", "major"]],
            on="student_id",
            how="left",
            validate="many_to_one",
        )
        [["student_id", "major", "wave", "study_hours", "quiz_score"]]
        .rename(columns={"quiz_score": "python_score"})
    )
    print("轮次表来源：E11 个人成果")
else:
    if USE_STUDENT_E11_OUTPUT:
        print("未找到 E11 个人成果，改用官方轮次分析视图。")
    wave_df = pd.read_csv(official_wave_path)
    print("轮次表来源：官方分析视图")

student_checks = {
    "学生表字段正确": student_df.columns.tolist() == student_columns,
    "学生表为 160 行": len(student_df) == 160,
    "student_id 唯一": student_df["student_id"].is_unique,
    "学生表无缺失": not student_df.isna().any().any(),
}
wave_checks = {
    "轮次表为 640 行": len(wave_df) == 640,
    "每名学生 4 轮": (
        wave_df.groupby("student_id")["wave"].size().eq(4).all()
    ),
    "复合主键唯一": not wave_df.duplicated(
        ["student_id", "wave"]
    ).any(),
    "四个轮次齐全": set(wave_df["wave"]) == {1, 2, 3, 4},
    "轮次成绩保留 11 个缺失": (
        wave_df["python_score"].isna().sum() == 11
    ),
}
input_checks = {**student_checks, **wave_checks}
for check_name, passed in input_checks.items():
    print(check_name, "：", passed)
if not all(input_checks.values()):
    failed = [name for name, passed in input_checks.items() if not passed]
    raise ValueError("输入核验未通过：" + "、".join(failed))

student_df["major"] = pd.Categorical(
    student_df["major"],
    categories=major_order,
    ordered=True,
)
wave_df["major"] = pd.Categorical(
    wave_df["major"],
    categories=major_order,
    ordered=True,
)
print("学生表形状：", student_df.shape)
print("轮次表形状：", wave_df.shape)

## 任务 1：用散点图探索学习时长与成绩

使用 `student_df` 绘制 `study_hours` 与 `python_score` 的散点图，并用 `major` 映射颜色。设置 `hue_order=major_order`，补充标题、坐标轴和图例标题。描述总体方向、离散程度和分组重叠，不根据颜色分组作因果判断。

In [ ]:
scatter_fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=student_df,
    x=...,
    y=...,
    hue=...,
    hue_order=major_order,
    alpha=0.75,
    ax=ax,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
ax.legend(title=..., bbox_to_anchor=(1.02, 1), loc="upper left")
scatter_fig.tight_layout()
plt.show()

## 任务 2：添加描述性趋势并量化线性相关

使用 `sns.regplot()` 绘制不分组的散点图和线性趋势线，同时计算 Pearson 相关系数。趋势线和相关系数用于概括当前样本的线性关联；本任务不进行显著性检验，也不能证明增加学习时长必然导致成绩提高。

In [ ]:
trend_fig, ax = plt.subplots(figsize=(7, 4))
sns.regplot(
    data=student_df,
    x=...,
    y=...,
    scatter_kws={"alpha": 0.45},
    line_kws={"color": "#E15759"},
    ax=ax,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
)
trend_fig.tight_layout()
plt.show()

study_score_corr = student_df[
    [...]
].corr().loc[..., ...]
print("学习时长与成绩的相关系数：", round(study_score_corr, 3))

## 任务 3：先汇总，再绘制有序变化

使用 `wave_df` 按 `wave` 和 `major` 分组，计算：

- 平均学习时长；
- 平均阶段成绩；
- 成绩的有效样本量。

再绘制各专业平均阶段成绩随轮次变化的折线图。横轴 `wave` 有明确顺序，因此可以连接；专业类别本身没有连续顺序，不能把专业名称放在横轴后随意连线。纵轴统一设置为 60–100 分，解释时还应报告实际均值或分差，不能只根据线条斜率判断变化大小。

In [ ]:
wave_summary = (
    wave_df.groupby(
        [...],
        observed=True,
        as_index=False,
    )
    .agg(
        mean_study_hours=(..., ...),
        mean_python_score=(..., ...),
        valid_scores=(..., ...),
    )
    .round(2)
)
print(wave_summary)

wave_fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(
    data=wave_summary,
    x=...,
    y=...,
    hue=...,
    hue_order=major_order,
    marker="o",
    ax=ax,
)
ax.set(
    title=...,
    xlabel=...,
    ylabel=...,
    xticks=[1, 2, 3, 4],
    ylim=(60, 100),
)
ax.legend(title=..., bbox_to_anchor=(1.02, 1), loc="upper left")
wave_fig.tight_layout()
plt.show()

## 任务 4：使用热力图展示相关结构

选择四个数值变量，计算相关矩阵并绘制热力图。设置：

- `annot=True` 显示数值；
- `fmt=".2f"` 保留两位小数；
- `vmin=-1`、`vmax=1` 保持相关系数的完整色标范围；
- 使用以 0 为中心的发散色板。

热力图用于快速发现值得继续研究的线性关系，不替代散点图和数据背景检查。

In [ ]:
numeric_columns = [...]
correlation_matrix = student_df[numeric_columns].corr()
print(correlation_matrix.round(2))

heatmap_fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    correlation_matrix,
    annot=...,
    fmt=...,
    cmap=...,
    center=0,
    vmin=...,
    vmax=...,
    square=True,
    ax=ax,
)
ax.set(title=...)
heatmap_fig.tight_layout()
plt.show()

## 任务 5：导出三张 EDA 图并完成核验

本次短报告使用任务 1、3、4 已完成的散点图、轮次图和热力图，不再从零重画。把它们分别导出到 `e14_output`，并运行检查代码。导出成功不代表解释正确，仍需在下一任务说明数据事实和边界。

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
figure_paths = {
    "scatter": output_dir / "e14_scatter.png",
    "wave": output_dir / "e14_wave.png",
    "heatmap": output_dir / "e14_heatmap.png",
}
scatter_fig.savefig(
    figure_paths["scatter"], dpi=150, bbox_inches="tight"
)
wave_fig.savefig(
    figure_paths["wave"], dpi=150, bbox_inches="tight"
)
heatmap_fig.savefig(
    figure_paths["heatmap"], dpi=150, bbox_inches="tight"
)

final_checks = {
    "三张图均已导出": all(path.exists() for path in figure_paths.values()),
    "相关系数处于合理范围": -1 <= study_score_corr <= 1,
    "汇总包含 12 个专业—轮次组合": len(wave_summary) == 12,
    "每组有效样本量已保留": wave_summary["valid_scores"].gt(0).all(),
}
for check_name, passed in final_checks.items():
    print(check_name, "：", passed)
if not all(final_checks.values()):
    raise ValueError("最终核验未通过。")
for label, path in figure_paths.items():
    print(label, "：", path)

## 任务 6：完成“三张图 + 一段解释”的 EDA 短报告

用 Markdown 按以下顺序表达：

1. **问题与数据范围**：说明研究问题、两张表的观测单位和样本范围。
2. **图形事实**：分别概括散点图、轮次图和热力图直接显示的现象。
3. **解释边界**：说明哪些结论不能由这些图证明。
4. **后续问题**：提出一个可在第六章使用两组比较或简单回归继续分析的问题。

### EDA 短报告

**问题与数据范围**

...

**图形事实**

...

**解释边界**

...

**后续问题**

...

## 任务 7：实验小结与工具核验

用 Markdown 回答：

1. 为什么 `student_df` 和 `wave_df` 不能直接按行混合解释？
2. 为什么轮次适合用折线图，而专业类别不适合随意连线？
3. 为什么相关热力图不能替代散点图？
4. 本实验是否使用了帮助文档、代码补全或 AI 工具？你如何核验其图形选择和文字解释？

### 实验小结与工具核验

- ...
- ...
- ...
- ...